# Session Distance Comparison

Calculate the average session distance traveled by Google Fit users using GPS data for two scenarios:
 Considering Earth's curvature (Haversine formula).
 Assuming a flat surface.
For each session, use the distance between the highest and lowest step IDs, and ignore sessions with only one step. Calculate and output the average distance for both scenarios and the difference between them.

Formulas:
1. Curved Earth: d=6371×arccos(sin(ϕ1​)×sin(ϕ2​)+cos(ϕ1​)×cos(ϕ2​)×cos(λ2​−λ1​))
2. Flat Surface: d=111×(lat2​−lat1​)2+(lon2​−lon1​)2​

🔍By solving this, you'll learn how to use CTE, Group By, Having, Case. Give it a try and share the output! 👇

In [0]:
CREATE TABLE ska_catalog.bronze.google_fit_location (user_id VARCHAR(50),session_id INT,step_id INT,day INT,latitude FLOAT,longitude FLOAT,altitude FLOAT);

INSERT INTO ska_catalog.bronze.google_fit_location (user_id, session_id, step_id, day, latitude, longitude, altitude)VALUES('user_1', 101, 1, 1, 37.7749, -122.4194, 15.0),('user_1', 101, 2, 1, 37.7750, -122.4195, 15.5),('user_1', 101, 3, 1, 37.7751, -122.4196, 16.0),('user_1', 102, 1, 1, 34.0522, -118.2437, 20.0),('user_1', 102, 2, 1, 34.0523, -118.2438, 20.5),('user_2', 201, 1, 1, 40.7128, -74.0060, 5.0),('user_2', 201, 2, 1, 40.7129, -74.0061, 5.5),('user_2', 202, 1, 1, 51.5074, -0.1278, 10.0),('user_2', 202, 2, 1, 51.5075, -0.1279, 10.5),('user_3', 301, 1, 1, 48.8566, 2.3522, 25.0),('user_3', 301, 2, 1, 48.8567, 2.3523, 25.5);

𝐄𝐱𝐩𝐥𝐚𝐧𝐚𝐭𝐢𝐨𝐧 𝐭𝐨 𝐒𝐨𝐥𝐯𝐞 𝐐𝐮𝐞𝐫𝐲
1. SessionMinMax CTE: It fetches the minimum and maximum step locations (start_lat, start_lon, end_lat, end_lon) for each session, filtering out sessions with only one step.

2. Distances CTE: Calculates the session distances using two methods:
Curved Surface: Using the Haversine formula to account for earth's curvature. Flat Surface: Using a simple Pythagorean formula assuming a flat surface.

3. Final Select Statement: It computes the average of the curved and flat distances and the difference between these averages.


In [0]:
SELECT * FROM ska_catalog.bronze.google_fit_location

In [0]:
SELECT
    user_id,
    session_id,
    day,
    MIN(CASE WHEN step_id = MIN(step_id)OVER (PARTITION BY user_id, session_id, day)THEN latitude END) AS start_lat,
    MIN(CASE WHEN step_id = MIN(step_id) OVER (PARTITION BY user_id, session_id, day) THEN longitude END ) AS start_lon,
    MAX(CASE WHEN step_id = MAX(step_id) OVER (PARTITION BY user_id, session_id, day) THEN latitude END ) AS ens_lat,
    MAX(CASE WHEN step_id = MAX(step_id) OVER (PARTITION BY user_id, session_id, day) THEN longitude END) AS end_lon
  FROM
    ska_catalog.bronze.google_fit_location
  GROUP BY
    user_id, session_id, day
  HAVING
    COUNT(DISTINCT step_id) > 1

In [0]:
WITH SessionMinMax AS (
  SELECT
    user_id,
    session_id,
    day,
    FIRST(latitude) OVER (PARTITION BY user_id, session_id, day ORDER BY step_id ASC) AS start_lat,
    FIRST(longitude) OVER (PARTITION BY user_id, session_id, day ORDER BY step_id ASC) AS start_lon,
    FIRST(latitude) OVER (PARTITION BY user_id, session_id, day ORDER BY step_id DESC) AS end_lat,
    FIRST(longitude) OVER (PARTITION BY user_id, session_id, day ORDER BY step_id DESC) AS end_lon
  FROM
    ska_catalog.bronze.google_fit_location
)
, FilteredSessions AS (
  SELECT
    user_id, session_id, day, start_lat, start_lon, end_lat, end_lon
  FROM
    SessionMinMax
  GROUP BY
    user_id, session_id, day, start_lat, start_lon, end_lat, end_lon
  HAVING
    COUNT(*) > 1
)
, distance AS (
  SELECT 
    user_id, session_id, day,
    6371 * ACOS(
      LEAST(
        1.0,
        SIN(RADIANS(start_lat)) * SIN(RADIANS(end_lat)) +
        COS(RADIANS(start_lat)) * COS(RADIANS(end_lat)) * COS(RADIANS(end_lon - start_lon))
      )
    ) AS distance_curved,
    SQRT(POWER(end_lat - start_lat, 2) + POWER(end_lon - start_lon, 2)) * 111 AS distance_flat
  FROM
    FilteredSessions
)
SELECT 
  AVG(distance_curved) AS avg_distance_curved,
  AVG(distance_flat) AS avg_distance_flat,
  AVG(distance_curved) - AVG(distance_flat) AS difference
FROM distance